# ⚡ Otimização de Hiperparâmetros - LightGBM
Este notebook apresenta o processo de ajuste fino do modelo **LightGBM (LGBM)** para o *Diabetes Prediction Challenge*. O foco aqui é a eficiência computacional e a exploração de diferentes estratégias de busca para lidar com o grande volume de dados (700k registros).

## 🛠️ Estratégia de Modelagem
* **Pipeline de Dados:** Utilização do preprocessador `v3a`, focado em otimizar variáveis para modelos baseados em gradiente.
* **Métrica de Sucesso:** Foco em **ROC-AUC**, utilizando o parâmetro `eval_metric='auc'` nativo do LightGBM para monitoramento em tempo real.
* **Desempenho:** Configuração de `force_row_wise=True` e `n_jobs=-1` para maximizar a velocidade de treinamento em máquinas multicore.

## 🔍 Fluxo de Otimização (Três Frentes de Busca)
O notebook explora três metodologias distintas para encontrar o conjunto ideal de hiperparâmetros:

1.  **Fase 3.1 - RandomizedSearchCV + Early Stopping:**
    * Implementa um *split* de validação dedicado (10% do treino).
    * Utiliza `callbacks` para interromper o treinamento caso a performance não melhore em 50 rodadas, economizando tempo e evitando *overfitting*.
2.  **Fase 3.2 - HalvingRandomSearchCV (Busca por Descarte):**
    * Uma estratégia de "torneio" onde candidatos a hiperparâmetros competem em subamostras dos dados.
    * Apenas os melhores candidatos avançam para as fases seguintes com mais recursos (amostras), permitindo explorar um espaço de parâmetros muito maior que a busca aleatória tradicional.
3.  **Fase 3.3 - Abordagem Híbrida (Halving + Early Stopping):**
    * Combina a eficiência do descarte sucessivo do *Halving* com a segurança do *Early Stopping*.
    * É a estratégia mais sofisticada do notebook, visando convergência rápida para o ótimo global com o menor custo computacional possível.

## 📊 Critérios de Sucesso
* **Controle de Complexidade:** Ajuste fino de `num_leaves` e `max_depth` para equilibrar o viés e a variância.
* **Regularização:** Exploração intensa de `reg_alpha` (L1) e `reg_lambda` (L2) para manter o modelo robusto a ruídos nos dados sintéticos.
* **Estabilidade:** Validação final via `evaluate_model` comparando as três estratégias de busca contra o Baseline.


In [1]:
import warnings
warnings.simplefilter("ignore", FutureWarning)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# LightGBM
from lightgbm import LGBMClassifier,early_stopping, log_evaluation


# sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv 

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    RandomizedSearchCV, StratifiedKFold,train_test_split,HalvingRandomSearchCV
)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)


from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import randint, uniform, loguniform

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 18:58:54


## 1. Load Data & Pipeline

In [2]:
BASE = Path.cwd().parent

PP3 = joblib.load(BASE/'src'/'preprocess_diabetes_v3a.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_val  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_val  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
mtd_scoring='roc_auc'
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("\n#Processo em:", time.strftime("%H:%M:%S"))


#Processo em: 18:58:55


## 2.Baseline

In [3]:
# MODEL BASE

print("#Processo iniciado em:", time.strftime("%H:%M:%S"))
model_base = LGBMClassifier(objective="binary",n_jobs=-1,random_state=42,verbose=-1,
                            force_row_wise=True)

pipe_base  = pipe_models(model_base, PP3)

R0=evaluate_model(pipe_base,X_train,y_train,X_val,y_val,modelname='Baseline')   

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo iniciado em: 18:58:55

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7201 ± 0.0010

✅ TEST SET
   Padrão (0.5):                  0.6808
   Otimizado:                     0.6809 (threshold = 0.510)
   ROC-AUC:                       0.7204
   Avg precision:                 0.8062
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Boosting Type             : gbdt
• Class Weight              : None
• Colsample Bytree          : 1.0
• Importance Type           : split
• Learning Rate             : 0.1
• Max Depth                 : -1
• Min Child Samples         : 20
• Min Child Weight          : 0.001
• Min Split Gain            : 0.0
• N Estimators              : 100
• N Jobs                    : -1
• Num Leaves                : 31
• Objective                 : binary
• Random State              : 42
• Reg Alpha                 : 0.0
• Reg Lambda                : 0.0
• Subs

## 3.Buscas por hiperparamentros
### 3.1 Early_stopping

In [4]:
# EARLY STOPPING
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# =====================================================
# a) Split de validação
# =====================================================
# 90% do treino é usando e 10% é usado valida o processo de stopping_rounds 
# X_search é o novo X_train
X_search, X_vals, y_search, y_vals = train_test_split(X_train, y_train, test_size=0.10, stratify=y_train)
X_val_p = PP3.transform(X_vals) # ajustar o processamento do x de validação

# =====================================================
# b) callbacks do LightGBM
# =====================================================
callbacks = [
    early_stopping(stopping_rounds=50,verbose=False), 
    log_evaluation(period=0)]


# =====================================================
# c) Definição do hiperparametro de Busca
# =====================================================
param_dist_1 = {
    # Boosting
    'model__n_estimators': randint(100, 6000),
    'model__learning_rate': loguniform(0.005, 0.3),

    # Estrutura
    'model__max_depth': randint(3, 10),
    'model__num_leaves': randint(40, 200),

    # Regularização
    'model__min_child_samples': randint(10, 100),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.1, 0.9),
    'model__reg_alpha': uniform(0, 5),
    'model__reg_lambda': [0, 0.5, 1, 3, 5, 10]
}

# =====================================================
# d) Definição do Buscador 
# =====================================================
numero_itera = 50
search_e = RandomizedSearchCV(
    pipe_base,
    param_dist_1,
    n_iter=numero_itera,
    cv=cv_s,
    scoring=mtd_scoring,
    random_state=42,
    verbose=0)

# =====================================================
# e) Buscando melhor conjunto de parametros 
# =====================================================
start = time.time()
search_e.fit(
    X_search, y_search,
    model__eval_set=[(X_val_p, y_vals)],
    model__eval_metric='auc',
    model__callbacks=callbacks)
end = time.time()

joblib.dump(search_e, 'search_random_earlystp_v3a.joblib') # salvar o buscador 

# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
best_e = search_e.best_estimator_

R1=evaluate_model(best_e,X_train,y_train,X_val,y_val,modelname='search 1',pipe_fit=False)   

print(f"{'='*70}")
print("🎯 Melhores Parâmetros Encontrados:")
print_hiper(search_e.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo iniciado em: 18:59:19

                         📍 RESULTADOS SEARCH 1                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7204 ± 0.0012

✅ TEST SET
   Padrão (0.5):                  0.6863
   Otimizado:                     0.6859 (threshold = 0.510)
   ROC-AUC:                       0.7278
   Avg precision:                 0.8119
🎯 Melhores Parâmetros Encontrados:
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample Bytree          : 0.14180537144799796
• Learning Rate             : 0.06015568027492316
• Max Depth                 : 7
• Min Child Samples         : 82
• N Estimators              : 5493
• Num Leaves                : 171
• Reg Alpha                 : 4.711008778424263
• Reg Lambda                : 10
• Subsample                 : 0.9233589392465844
--------------------------------------------------

#Processo finalizado em: 20:10:09


### 3.2 HalvingRandomSearchCV

In [5]:
print("# Processo  iniciado em:", time.strftime("%H:%M:%S"))

# HalvingRandomSearchCV
# =====================================================
# c) Definição do hiperparametro de Busca
# =====================================================
param_dist_1 = {
    # Boosting
    'model__n_estimators': randint(100, 3000),
    'model__learning_rate': loguniform(0.005, 0.3),

    # Estrutura
    'model__max_depth': randint(3, 10),
    'model__num_leaves': randint(40, 200),

    # Regularização
    'model__min_child_samples': randint(10, 100),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.1, 0.9),
    'model__reg_alpha': uniform(0, 5),
    'model__reg_lambda': [0, 0.5, 1, 3, 5, 10]
}
#-------------------------------------------------------
# d) Definição do Buscador 
#-------------------------------------------------------
search_h = HalvingRandomSearchCV(
    pipe_base,
    param_dist_1,
    resource='n_samples',           
    max_resources='auto',
    min_resources='exhaust',             
    factor=3,                    
    n_candidates=81,            
    cv=cv_s,
    scoring=mtd_scoring,
    random_state=42,
    verbose=0,
    n_jobs=2                       
)

#-------------------------------------------------------
# e) Buscando melhor conjunto de parametros 
#-------------------------------------------------------
start = time.time()
search_h.fit(X_train, y_train)
end = time.time()
joblib.dump(search_h, 'search_Hrandom_v3a.joblib')


best_h = search_h.best_estimator_

R2=evaluate_model(best_h,X_train,y_train,X_val,y_val,modelname='search 2',pipe_fit=False)   

print(f"{'='*70}")
print("🎯 Melhores Parâmetros Encontrados:")
print_hiper(search_h.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

# Processo  iniciado em: 20:10:09

                         📍 RESULTADOS SEARCH 2                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7257 ± 0.0011

✅ TEST SET
   Padrão (0.5):                  0.6837
   Otimizado:                     0.6838 (threshold = 0.520)
   ROC-AUC:                       0.7255
   Avg precision:                 0.8103
🎯 Melhores Parâmetros Encontrados:
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample Bytree          : 0.14070456001948428
• Learning Rate             : 0.01894333932851415
• Max Depth                 : 7
• Min Child Samples         : 91
• N Estimators              : 978
• Num Leaves                : 92
• Reg Alpha                 : 2.9337558283192413
• Reg Lambda                : 0
• Subsample                 : 0.7123738038749523
--------------------------------------------------

#Processo finalizado em: 21:07:50


### 3.3 HalvingRandomSearchCV + early stopping

In [11]:
# # HalvingRandomSearchCV+ early stopping
print("# Processo iniciado em:", time.strftime("%H:%M:%S"))

# =====================================================
# a) Split de validação
# =====================================================
# 90% do treino é usando e 10% é usado valida o processo de stopping_rounds 
# X_search é o novo X_train
X_search, X_vals, y_search, y_vals = train_test_split(X_train, y_train, test_size=0.10, stratify=y_train)
X_val_p = PP3.transform(X_vals) # ajustar o processamento do x de validação

# =====================================================
# b) callbacks do LightGBM
# =====================================================
callbacks = [
    early_stopping(stopping_rounds=50,verbose=False), 
    log_evaluation(period=0)]

# =====================================================
# c) Definição do hiperparametro de Busca
# =====================================================
param_dist_1 = {
    # Boosting
    'model__n_estimators': randint(100, 3000),
    'model__learning_rate': loguniform(0.005, 0.3),

    # Estrutura
    'model__max_depth': randint(3, 10),
    'model__num_leaves': randint(40, 200),

    # Regularização
    'model__min_child_samples': randint(10, 100),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.1, 0.9),
    'model__reg_alpha': uniform(0, 5),
    'model__reg_lambda': [0, 0.5, 1, 3, 5, 10]
}
#-------------------------------------------------------
# d) Definição do Buscador 
#-------------------------------------------------------
search_h2 = HalvingRandomSearchCV(
    pipe_base,
    param_dist_1,
    resource='n_samples',           
    max_resources='auto',
    min_resources='exhaust',             
    factor=3,                    
    n_candidates=81,            
    cv=cv_s,
    scoring=mtd_scoring,
    random_state=42,
    verbose=0,
    n_jobs=-1                       
)

#-------------------------------------------------------
# e) Buscando melhor conjunto de parametros 
#-------------------------------------------------------
start = time.time()
search_h2.fit(
    X_search, y_search,
    model__eval_set=[(X_val_p, y_vals)],
    model__eval_metric='auc',    #AUC apenas aqui
    model__callbacks=callbacks)
end = time.time()

joblib.dump(search_h2, 'search_Hrandom_v3a.joblib')

best_h2 = search_h2.best_estimator_
R3=evaluate_model(best_h2,X_train,y_train,X_val,y_val,modelname='search 3',pipe_fit=False)   

print(f"{'='*70}")
print("🎯 Melhores Parâmetros Encontrados:")
print_hiper(search_h2.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


# Processo iniciado em: 21:47:25

                         📍 RESULTADOS SEARCH 3                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7281 ± 0.0012

✅ TEST SET
   Padrão (0.5):                  0.6858
   Otimizado:                     0.6850 (threshold = 0.520)
   ROC-AUC:                       0.7278
   Avg precision:                 0.8119
🎯 Melhores Parâmetros Encontrados:
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample Bytree          : 0.15180288401497988
• Learning Rate             : 0.047436809713707416
• Max Depth                 : 7
• Min Child Samples         : 44
• N Estimators              : 1338
• Num Leaves                : 114
• Reg Alpha                 : 0.5853350821380293
• Reg Lambda                : 3
• Subsample                 : 0.8752120040700155
--------------------------------------------------

#Processo finalizado em: 22:32:45


In [12]:
end_final = time.time()
print(f"Tempo final: {(end_final-start_inicial)/60:.2f} min")

Tempo final: 213.84 min


In [13]:
# Salvar Hiperparametros joblib
joblib.dump(pipe_base, 'modelo_LGBM2_final_base.'+mtd_scoring+'_v3a.joblib')
joblib.dump(search_e.best_estimator_, 'modelo_LGBM2_final_randsearch_e.'+mtd_scoring+'_v3a.joblib')
joblib.dump(search_h.best_estimator_, 'modelo_LGBM2_final_Hrandsearch.'+mtd_scoring+'_v3a.joblib')
joblib.dump(search_h2.best_estimator_, 'modelo_LGBM2_final_Hrandsearch_e.'+mtd_scoring+'_v3a.joblib')
print(f"✅ Arquivo salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo salvo • 10/03/2026-22:32:45
